In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle
import json
from tqdm import tqdm

# Data paths
CSV_FILE = 'dataset/dataset.csv'
VIDEO_DIR = 'dataset/dynamic'
OUTPUT_DIR = 'processed_features'

# Feature extraction settings
SEQUENCE_LENGTH = 30
TEST_SIZE = 0.15
VAL_SIZE = 0.15
RANDOM_STATE = 42
FEATURE_DIM = 258

print("="*70)
print("FEATURE EXTRACTION AND AUGMENTATION")
print("="*70)
print(f"CSV file: {CSV_FILE}")
print(f"Video directory: {VIDEO_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Sequence length: {SEQUENCE_LENGTH}")
print(f"Feature dimension: {FEATURE_DIM}")
print("="*70)

os.makedirs(OUTPUT_DIR, exist_ok=True)

FEATURE EXTRACTION AND AUGMENTATION
CSV file: dataset/dataset.csv
Video directory: dataset/dynamic
Output directory: processed_features
Sequence length: 30
Feature dimension: 258


In [2]:
print("\nLoading dataset...")

df = pd.read_csv(CSV_FILE, encoding='utf-8')
print(f"Loaded {len(df)} entries from CSV")

# Validate video paths
valid_indices = []
for idx, row in df.iterrows():
    if os.path.exists(row['file']):
        valid_indices.append(idx)

df_valid = df.iloc[valid_indices].reset_index(drop=True)
print(f"Found {len(df_valid)} valid videos")

# Extract paths and labels
video_paths = df_valid['file'].values
labels = df_valid['gesture'].values

print("\nSplitting data...")

# First split: separate test set
X_temp, X_test_paths, y_temp, y_test = train_test_split(
    video_paths, labels,
    test_size=TEST_SIZE,
    stratify=labels,
    random_state=RANDOM_STATE
)

# Second split: separate train and validation
X_train_paths, X_val_paths, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print(f"\nSplit complete:")
print(f"   Train: {len(X_train_paths)}")
print(f"   Val:   {len(X_val_paths)}")
print(f"   Test:  {len(X_test_paths)}")

# Verify no overlap
train_set = set(X_train_paths)
val_set = set(X_val_paths)
test_set = set(X_test_paths)

if len(train_set & val_set) == 0 and len(train_set & test_set) == 0:
    print(f"   No data leakage detected")


Loading dataset...
Loaded 670 entries from CSV
Found 670 valid videos

Splitting data...

Split complete:
   Train: 468
   Val:   101
   Test:  101
   No data leakage detected


In [3]:
def adjust_brightness(frame, factor):
    """Adjust brightness of frame"""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    v = np.clip(v * factor, 0, 255).astype(np.uint8)
    hsv = cv2.merge([h, s, v])
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def mirror_frame(frame):
    """Horizontal flip of frame"""
    return cv2.flip(frame, 1)

print("Augmentation functions loaded")

Augmentation functions loaded


In [4]:
print("Initializing MediaPipe...")

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    model_complexity=0
)

def extract_landmarks_from_results(results):
    """Extract 258 features from MediaPipe results"""
    if not results.multi_hand_landmarks:
        return None
    
    all_features = []
    
    # Process up to 2 hands
    for i in range(2):
        if i < len(results.multi_hand_landmarks):
            hand_landmarks = results.multi_hand_landmarks[i]
            
            # Raw coordinates (63 features)
            raw_coords = []
            for lm in hand_landmarks.landmark:
                raw_coords.extend([lm.x, lm.y, lm.z])
            all_features.extend(raw_coords)
            
            # Normalized coordinates relative to wrist (63 features)
            wrist = hand_landmarks.landmark[0]
            norm_coords = []
            for lm in hand_landmarks.landmark:
                norm_coords.extend([
                    lm.x - wrist.x,
                    lm.y - wrist.y,
                    lm.z - wrist.z
                ])
            all_features.extend(norm_coords)
        else:
            # Pad with zeros if hand not detected (126 features)
            all_features.extend([0.0] * 126)
    
    # Additional features (6 features)
    num_hands = len(results.multi_hand_landmarks)
    all_features.extend([
        num_hands / 2.0,
        1.0 if num_hands > 0 else 0.0,
        1.0 if num_hands > 1 else 0.0,
        0.5,
        0.5,
        0.0
    ])
    
    return np.array(all_features, dtype=np.float32)

print("MediaPipe initialized")

Initializing MediaPipe...
MediaPipe initialized


In [5]:
def process_video(video_path, target_frames=SEQUENCE_LENGTH, resize_width=320):
    """Extract features from original video"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames >= target_frames:
        frame_indices = np.linspace(0, total_frames - 1, target_frames, dtype=int)
    else:
        frame_indices = range(total_frames)
    
    all_landmarks = []
    
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret:
            continue
        
        # Resize frame
        height, width = frame.shape[:2]
        new_height = int(height * (resize_width / width))
        frame_resized = cv2.resize(frame, (resize_width, new_height))
        
        # Process with MediaPipe
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)
        
        landmarks = extract_landmarks_from_results(results)
        if landmarks is not None:
            all_landmarks.append(landmarks)
    
    cap.release()
    
    if len(all_landmarks) == 0:
        return None
    
    # Pad or trim to target length
    if len(all_landmarks) >= target_frames:
        sequence = all_landmarks[:target_frames]
    else:
        sequence = all_landmarks.copy()
        while len(sequence) < target_frames:
            sequence.append(all_landmarks[-1])
        sequence = sequence[:target_frames]
    
    return np.array(sequence, dtype=np.float32)

def process_video_with_augmentation(video_path, aug_type, brightness_factor=None):
    """Process video with augmentation"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames >= SEQUENCE_LENGTH:
        frame_indices = np.linspace(0, total_frames - 1, SEQUENCE_LENGTH, dtype=int)
    else:
        frame_indices = range(total_frames)
    
    all_landmarks = []
    
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret:
            continue
        
        # Apply augmentation
        if aug_type == 'mirror':
            frame = mirror_frame(frame)
        elif aug_type == 'brightness' and brightness_factor:
            frame = adjust_brightness(frame, brightness_factor)
        
        # Resize and process
        height, width = frame.shape[:2]
        frame_resized = cv2.resize(frame, (320, int(height * (320 / width))))
        
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)
        
        landmarks = extract_landmarks_from_results(results)
        if landmarks is not None:
            all_landmarks.append(landmarks)
    
    cap.release()
    
    if len(all_landmarks) == 0:
        return None
    
    # Pad or trim
    if len(all_landmarks) >= SEQUENCE_LENGTH:
        sequence = all_landmarks[:SEQUENCE_LENGTH]
    else:
        sequence = all_landmarks.copy()
        while len(sequence) < SEQUENCE_LENGTH:
            sequence.append(all_landmarks[-1])
        sequence = sequence[:SEQUENCE_LENGTH]
    
    return np.array(sequence, dtype=np.float32)

print("Video processing functions loaded")

Video processing functions loaded


In [6]:
def extract_features_with_augmentation(paths, labels, set_name, augment=False):
    """Extract features with augmentation for training set only"""
    features = []
    labels_out = []
    failed = 0
    
    print(f"\n{set_name} set (augment={augment}):")
    
    for path, label in tqdm(zip(paths, labels), total=len(paths), desc="Processing"):
        # Original video
        sequence = process_video(path, SEQUENCE_LENGTH)
        if sequence is not None:
            features.append(sequence)
            labels_out.append(label)
        else:
            failed += 1
        
        # Add augmentations ONLY for training set
        if augment:
            # Mirror augmentation
            aug_seq = process_video_with_augmentation(path, 'mirror')
            if aug_seq is not None:
                features.append(aug_seq)
                labels_out.append(label)
            
            # Darker (brightness 0.8)
            aug_seq = process_video_with_augmentation(path, 'brightness', 0.8)
            if aug_seq is not None:
                features.append(aug_seq)
                labels_out.append(label)
            
            # Brighter (brightness 1.2)
            aug_seq = process_video_with_augmentation(path, 'brightness', 1.2)
            if aug_seq is not None:
                features.append(aug_seq)
                labels_out.append(label)
    
    print(f"   Extracted: {len(features)} (Failed: {failed})")
    if augment:
        print(f"   Augmentation: {len(paths)} -> {len(features)} ({len(features)/len(paths):.1f}x)")
    
    return np.array(features, dtype=np.float32), np.array(labels_out)

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(labels)

y_train_encoded = label_encoder.transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# Extract features
print("\nExtracting features...")

X_train_features, y_train_final = extract_features_with_augmentation(
    X_train_paths, y_train_encoded, "Train", augment=True
)

X_val_features, y_val_final = extract_features_with_augmentation(
    X_val_paths, y_val_encoded, "Validation", augment=False
)

X_test_features, y_test_final = extract_features_with_augmentation(
    X_test_paths, y_test_encoded, "Test", augment=False
)

print(f"\nFeature extraction complete:")
print(f"   Train shape: {X_train_features.shape}")
print(f"   Val shape:   {X_val_features.shape}")
print(f"   Test shape:  {X_test_features.shape}")

# Close MediaPipe
hands.close()


Extracting features...

Train set (augment=True):


Processing: 100%|████████████████████████████████████████████████████████████████████| 468/468 [31:28<00:00,  4.04s/it]


   Extracted: 1872 (Failed: 0)
   Augmentation: 468 -> 1872 (4.0x)

Validation set (augment=False):


Processing: 100%|████████████████████████████████████████████████████████████████████| 101/101 [02:01<00:00,  1.20s/it]


   Extracted: 101 (Failed: 0)

Test set (augment=False):


Processing: 100%|████████████████████████████████████████████████████████████████████| 101/101 [02:03<00:00,  1.22s/it]

   Extracted: 101 (Failed: 0)

Feature extraction complete:
   Train shape: (1872, 30, 258)
   Val shape:   (101, 30, 258)
   Test shape:  (101, 30, 258)


In [7]:
print("\nSaving extracted features...")

# Save features
np.save(os.path.join(OUTPUT_DIR, 'X_train.npy'), X_train_features)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train_final)
np.save(os.path.join(OUTPUT_DIR, 'X_val.npy'), X_val_features)
np.save(os.path.join(OUTPUT_DIR, 'y_val.npy'), y_val_final)
np.save(os.path.join(OUTPUT_DIR, 'X_test.npy'), X_test_features)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test_final)

print(f"Features saved to {OUTPUT_DIR}")
print(f"   X_train.npy: {X_train_features.shape}")
print(f"   y_train.npy: {y_train_final.shape}")
print(f"   X_val.npy: {X_val_features.shape}")
print(f"   y_val.npy: {y_val_final.shape}")
print(f"   X_test.npy: {X_test_features.shape}")
print(f"   y_test.npy: {y_test_final.shape}")

# Save label encoder
label_encoder_path = os.path.join(OUTPUT_DIR, 'label_encoder.pkl')
with open(label_encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"\nLabel encoder saved: {label_encoder_path}")

# Save metadata
metadata = {
    'sequence_length': SEQUENCE_LENGTH,
    'feature_dim': FEATURE_DIM,
    'train_samples': len(X_train_features),
    'val_samples': len(X_val_features),
    'test_samples': len(X_test_features),
    'augmentation_factor': len(X_train_features) / len(X_train_paths),
    'num_classes': len(label_encoder.classes_),
    'class_names': label_encoder.classes_.tolist(),
    'augmentations': ['original', 'mirror', 'brightness_0.8', 'brightness_1.2'],
    'test_size': TEST_SIZE,
    'val_size': VAL_SIZE,
    'random_state': RANDOM_STATE
}

metadata_path = os.path.join(OUTPUT_DIR, 'metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print(f"Metadata saved: {metadata_path}")

print("\n" + "="*70)
print("FEATURE EXTRACTION COMPLETE")
print("="*70)
print(f"All files saved in: {OUTPUT_DIR}")
print("="*70)


Saving extracted features...
Features saved to processed_features
   X_train.npy: (1872, 30, 258)
   y_train.npy: (1872,)
   X_val.npy: (101, 30, 258)
   y_val.npy: (101,)
   X_test.npy: (101, 30, 258)
   y_test.npy: (101,)

Label encoder saved: processed_features\label_encoder.pkl
Metadata saved: processed_features\metadata.json

FEATURE EXTRACTION COMPLETE
All files saved in: processed_features
